##LANDING → BUSINESS-KEY DEDUPLICATION
BUSINESS-KEY DEDUPLICATION  write to bronze table

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# CONFIGURATION
# ============================================================

catalog = "vb_ecommerce"

landing_base = (
    "abfss://landing@adlsvbecommercedev.dfs.core.windows.net/"
    "ecommerce/sql"
)

tables_config = {
    "customers": "CustomerID",
    "products": "ProductID",
    "stores": "StoreID",
    "salesorders": "OrderID",
    "salesorderitems": "OrderItemID",
    "payments": "PaymentID"
}

# ============================================================
# LANDING → BUSINESS-KEY DEDUPLICATION
# ============================================================

dedup_dfs = {}

print("=" * 70)
print("LANDING → BUSINESS-KEY DEDUPLICATION")
print("=" * 70)

for table, business_key in tables_config.items():

    source_path = f"{landing_base}/{table}"

    print(f"\nProcessing: {table}")
    print(f"Business Key: {business_key}")

    # --------------------------------------------------------
    # Read all historical Landing files
    # --------------------------------------------------------

    df = (
        spark.read
        .option("recursiveFileLookup", "true")
        .parquet(source_path)
    )

    print(f"Landing records: {df.count()}")

    # --------------------------------------------------------
    # Keep latest record for each business key
    # --------------------------------------------------------

    window_spec = (
        Window
        .partitionBy(business_key)
        .orderBy(F.col("ModifiedDate").desc())
    )

    df_dedup = (
        df
        .withColumn("_rn", F.row_number().over(window_spec))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )

    final_count = df_dedup.count()

    print(f"Unique business records: {final_count}")

    dedup_dfs[table] = df_dedup

print("\n" + "=" * 70)
print("BUSINESS-KEY DEDUPLICATION COMPLETED")
print("=" * 70)

LANDING → BUSINESS-KEY DEDUPLICATION

Processing: customers
Business Key: CustomerID
Landing records: 58
Unique business records: 50

Processing: products
Business Key: ProductID
Landing records: 58
Unique business records: 50

Processing: stores
Business Key: StoreID
Landing records: 56
Unique business records: 50

Processing: salesorders
Business Key: OrderID
Landing records: 58
Unique business records: 50

Processing: salesorderitems
Business Key: OrderItemID
Landing records: 61
Unique business records: 50

Processing: payments
Business Key: PaymentID
Landing records: 58
Unique business records: 50

BUSINESS-KEY DEDUPLICATION COMPLETED


## =================================
# DEDUPLICATION VERIFICATION
# =================================

In [0]:
from pyspark.sql import functions as F

# ============================================================
# DEDUPLICATION VERIFICATION
# ============================================================

landing_base = (
    "abfss://landing@adlsvbecommercedev.dfs.core.windows.net/"
    "ecommerce/sql"
)

tables_config = {
    "customers": "CustomerID",
    "products": "ProductID",
    "stores": "StoreID",
    "salesorders": "OrderID",
    "salesorderitems": "OrderItemID",
    "payments": "PaymentID"
}

print("=" * 70)
print("DEDUPLICATION VERIFICATION")
print("=" * 70)

for table, business_key in tables_config.items():

    source_path = f"{landing_base}/{table}"

    df = (
        spark.read
        .option("recursiveFileLookup", "true")
        .parquet(source_path)
    )

    total_records = df.count()

    unique_keys = (
        df.select(business_key)
          .distinct()
          .count()
    )

    duplicate_records = total_records - unique_keys

    print(
        f"{table:<20} "
        f"Total: {total_records:<5} "
        f"Unique: {unique_keys:<5} "
        f"Duplicates: {duplicate_records}"
    )

DEDUPLICATION VERIFICATION
customers            Total: 58    Unique: 50    Duplicates: 8
products             Total: 58    Unique: 50    Duplicates: 8
stores               Total: 56    Unique: 50    Duplicates: 6
salesorders          Total: 58    Unique: 50    Duplicates: 8
salesorderitems      Total: 61    Unique: 50    Duplicates: 11
payments             Total: 58    Unique: 50    Duplicates: 8


## BUSINESS-KEY DEDUPLICATED DATA to SILVER

In [0]:
# ============================================================
# BUSINESS-KEY DEDUPLICATED DATA to SILVER
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = "vb_ecommerce"

landing_base = (
    "abfss://landing@adlsvbecommercedev.dfs.core.windows.net/"
    "ecommerce/sql"
)

tables_config = {
    "customers": "CustomerID",
    "products": "ProductID",
    "stores": "StoreID",
    "salesorders": "OrderID",
    "salesorderitems": "OrderItemID",
    "payments": "PaymentID"
}

print("=" * 70)
print("LANDING → BUSINESS-KEY DEDUPLICATION → SILVER")
print("=" * 70)

for table, business_key in tables_config.items():

    source_path = f"{landing_base}/{table}"
    target_table = f"{catalog}.silver.{table}"

    print(f"\nProcessing: {table}")

    # --------------------------------------------------------
    # 1. Read all Landing files
    # --------------------------------------------------------

    df = (
        spark.read
        .option("recursiveFileLookup", "true")
        .parquet(source_path)
    )

    # --------------------------------------------------------
    # 2. Keep latest record for each business key
    # --------------------------------------------------------

    window_spec = (
        Window
        .partitionBy(business_key)
        .orderBy(F.col("ModifiedDate").desc())
    )

    df_silver = (
        df
        .withColumn("_rn", F.row_number().over(window_spec))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )

    # --------------------------------------------------------
    # 3. Write to existing Silver table
    # --------------------------------------------------------

    (
        df_silver.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    print(f"Silver table : {target_table}")
    print(f"Silver rows  : {df_silver.count()}")

print("\n" + "=" * 70)
print("SILVER LOAD COMPLETED")
print("=" * 70)

LANDING → BUSINESS-KEY DEDUPLICATION → SILVER

Processing: customers
Silver table : vb_ecommerce.silver.customers
Silver rows  : 50

Processing: products
Silver table : vb_ecommerce.silver.products
Silver rows  : 50

Processing: stores
Silver table : vb_ecommerce.silver.stores
Silver rows  : 50

Processing: salesorders
Silver table : vb_ecommerce.silver.salesorders
Silver rows  : 50

Processing: salesorderitems
Silver table : vb_ecommerce.silver.salesorderitems
Silver rows  : 50

Processing: payments
Silver table : vb_ecommerce.silver.payments
Silver rows  : 50

SILVER LOAD COMPLETED


#Build the Gold layer

In [0]:
from pyspark.sql import functions as F

catalog = "vb_ecommerce"

# ============================================================
# GOLD LAYER - E-COMMERCE CURATED MODEL
# ============================================================

print("=" * 70)
print("SILVER → GOLD")
print("=" * 70)


# ============================================================
# 1. DIM CUSTOMER
# ============================================================
# Current customer state.
# SCD2 was already demonstrated separately.

df_customer = spark.table(f"{catalog}.silver.customers")

dim_customer = (
    df_customer
    .select(
        "CustomerID",
        "FirstName",
        "LastName",
        "Email",
        "Phone",
        "City",
        "State",
        "Country",
        "CreatedDate",
        "ModifiedDate",
        "IsDeleted"
    )
)

(
    dim_customer.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.gold.dim_customer")
)

print("dim_customer created")


# ============================================================
# 2. DIM PRODUCT
# ============================================================

df_product = spark.table(f"{catalog}.silver.products")

dim_product = (
    df_product
    .withColumn(
        "ProductStatus",
        F.when(F.col("IsDeleted") == True, "Inactive")
         .otherwise("Active")
    )
    .select(
        "ProductID",
        "ProductName",
        "Category",
        "Price",
        "StockQuantity",
        "ProductStatus",
        "CreatedDate",
        "ModifiedDate"
    )
)

(
    dim_product.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.gold.dim_product")
)

print("dim_product created")


# ============================================================
# 3. DIM STORE
# ============================================================

df_store = spark.table(f"{catalog}.silver.stores")

dim_store = (
    df_store
    .select(
        "StoreID",
        "StoreName",
        "City",
        "State",
        "Region",
        "CreatedDate",
        "ModifiedDate",
        "IsDeleted"
    )
)

(
    dim_store.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.gold.dim_store")
)

print("dim_store created")


# ============================================================
# 4. FACT ORDERS
# ============================================================

df_orders = spark.table(f"{catalog}.silver.salesorders")

fact_orders = (
    df_orders
    .select(
        "OrderID",
        "CustomerID",
        "StoreID",
        "OrderDate",
        "OrderStatus",
        "TotalAmount",
        "CreatedDate",
        "ModifiedDate",
        "IsDeleted"
    )
)

(
    fact_orders.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.gold.fact_orders")
)

print("fact_orders created")


# ============================================================
# 5. FACT ORDER ITEMS / SALES
# ============================================================

df_items = spark.table(f"{catalog}.silver.salesorderitems")

fact_sales = (
    df_items
    .select(
        "OrderItemID",
        "OrderID",
        "ProductID",
        "Quantity",
        "UnitPrice",
        "DiscountAmount",
        "CreatedDate",
        "ModifiedDate",
        "IsDeleted"
    )
    .withColumn(
        "GrossAmount",
        F.col("Quantity") * F.col("UnitPrice")
    )
    .withColumn(
        "NetAmount",
        (F.col("Quantity") * F.col("UnitPrice"))
        - F.coalesce(F.col("DiscountAmount"), F.lit(0))
    )
)

(
    fact_sales.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.gold.fact_sales")
)

print("fact_sales created")


# ============================================================
# 6. FACT PAYMENTS
# ============================================================

df_payments = spark.table(f"{catalog}.silver.payments")

fact_payments = (
    df_payments
    .select(
        "PaymentID",
        "OrderID",
        "CustomerID",
        "PaymentMethod",
        "PaymentAmount",
        "PaymentStatus",
        "PaymentDate",
        "CreatedDate",
        "ModifiedDate",
        "IsDeleted"
    )
)

(
    fact_payments.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.gold.fact_payments")
)

print("fact_payments created")


# ============================================================
# FINAL VALIDATION
# ============================================================

gold_tables = [
    "dim_customer",
    "dim_product",
    "dim_store",
    "fact_orders",
    "fact_sales",
    "fact_payments"
]

print("\n" + "=" * 70)
print("GOLD VALIDATION")
print("=" * 70)

for table in gold_tables:

    count = spark.table(
        f"{catalog}.gold.{table}"
    ).count()

    print(f"{table:<25} : {count} records")

print("\n" + "=" * 70)
print("SILVER → GOLD COMPLETED SUCCESSFULLY")
print("=" * 70)

SILVER → GOLD
dim_customer created
dim_product created
dim_store created
fact_orders created
fact_sales created
fact_payments created

GOLD VALIDATION
dim_customer              : 50 records
dim_product               : 50 records
dim_store                 : 50 records
fact_orders               : 50 records
fact_sales                : 50 records
fact_payments             : 50 records

SILVER → GOLD COMPLETED SUCCESSFULLY


##OPTIMIZE + ZORDER Gold

In [0]:
%sql
-- ============================================================
-- GOLD LAYER: OPTIMIZE + ZORDER
--   ============================================================

OPTIMIZE vb_ecommerce.gold.dim_customer
ZORDER BY (CustomerID);

OPTIMIZE vb_ecommerce.gold.dim_product
ZORDER BY (ProductID);

OPTIMIZE vb_ecommerce.gold.dim_store
ZORDER BY (StoreID);

OPTIMIZE vb_ecommerce.gold.fact_orders
ZORDER BY (CustomerID, StoreID);

OPTIMIZE vb_ecommerce.gold.fact_sales
ZORDER BY (ProductID, OrderID);

OPTIMIZE vb_ecommerce.gold.fact_payments
ZORDER BY (CustomerID, OrderID);

path,metrics
abfss://bronze@adlsvbecommercedev.dfs.core.windows.net/unitycatalog/__unitystorage/catalogs/246c5646-5879-4910-8630-869564b8c63e/tables/69fd0840-8892-42c5-b96a-679785c62bae,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 4685), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1788170174173, 1788170176771, 4, 0, null, List(0, 0), null, 10, 10, 0, 0, null)"


##Describe delta

In [0]:
%sql

DESCRIBE DETAIL vb_ecommerce.gold.dim_customer;

DESCRIBE DETAIL vb_ecommerce.gold.dim_product;

DESCRIBE DETAIL vb_ecommerce.gold.dim_store;

DESCRIBE DETAIL vb_ecommerce.gold.fact_orders;

DESCRIBE DETAIL vb_ecommerce.gold.fact_sales;

DESCRIBE DETAIL vb_ecommerce.gold.fact_payments;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,2efd452b-277e-4b0f-a90b-dd9b7ff790a2,vb_ecommerce.gold.fact_payments,null,abfss://bronze@adlsvbecommercedev.dfs.core.windows.net/unitycatalog/__unitystorage/catalogs/246c5646-5879-4910-8630-869564b8c63e/tables/69fd0840-8892-42c5-b96a-679785c62bae,2026-08-31T09:42:01.466Z,2026-08-31T09:42:03Z,List(),List(),1,4685,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


###Gold MERGE.

In [0]:
%sql

-- ============================================================
-- GOLD MERGE DEMONSTRATION
-- Update ProductID = 1 in the existing Gold table
-- ============================================================

MERGE INTO vb_ecommerce.gold.dim_product AS tgt
USING (
    SELECT
        1 AS ProductID,
        'Laptop' AS ProductName,
        'Electronics' AS Category,
        CAST(70000.00 AS DECIMAL(18,2)) AS Price,
        45 AS StockQuantity,
        'Active' AS ProductStatus,
        current_timestamp() AS CreatedDate,
        current_timestamp() AS ModifiedDate
) AS src

ON tgt.ProductID = src.ProductID

WHEN MATCHED THEN UPDATE SET
    tgt.ProductName  = src.ProductName,
    tgt.Category     = src.Category,
    tgt.Price        = src.Price,
    tgt.StockQuantity = src.StockQuantity,
    tgt.ProductStatus = src.ProductStatus,
    tgt.ModifiedDate = src.ModifiedDate

WHEN NOT MATCHED THEN INSERT (
    ProductID,
    ProductName,
    Category,
    Price,
    StockQuantity,
    ProductStatus,
    CreatedDate,
    ModifiedDate
)
VALUES (
    src.ProductID,
    src.ProductName,
    src.Category,
    src.Price,
    src.StockQuantity,
    src.ProductStatus,
    src.CreatedDate,
    src.ModifiedDate
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1,1,0,0


In [0]:
%sql

SELECT *
FROM vb_ecommerce.gold.dim_product
WHERE ProductID = 1;

ProductID,ProductName,Category,Price,StockQuantity,ProductStatus,CreatedDate,ModifiedDate
1,Laptop,Electronics,70000.00,45,Active,2026-08-28T22:23:46.173333Z,2026-08-31T10:00:08.365857Z


# Creating external storage for SILVER

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS extloc_vb_ecommerce_silver
URL 'abfss://silver@adlsvbecommercedev.dfs.core.windows.net/'
WITH (CREDENTIAL cred_vb_ecommerce_adls)
COMMENT 'External location for Silver curated data';

# Creating external storage for GOLD

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS extloc_vb_ecommerce_gold
URL 'abfss://gold@adlsvbecommercedev.dfs.core.windows.net/'
WITH (CREDENTIAL cred_vb_ecommerce_adls)
COMMENT 'External location for Gold curated data';

In [0]:
%sql SHOW EXTERNAL LOCATIONS;

name,url,comment
dbw_vb_ecommerce_dev,abfss://unity-catalog-storage@dbstoragexoyocruayeybq.dfs.core.windows.net/7405617028985756,null
extloc_vb_ecommerce,abfss://landing@adlsvbecommercedev.dfs.core.windows.net/,null
extloc_vb_ecommerce_gold,abfss://gold@adlsvbecommercedev.dfs.core.windows.net/,External location for Gold curated data
extloc_vb_ecommerce_silver,abfss://silver@adlsvbecommercedev.dfs.core.windows.net/,External location for Silver curated data
extloc_vb_ecommerce_uc,abfss://bronze@adlsvbecommercedev.dfs.core.windows.net/unitycatalog,bronze/unitycatalog


##6 Gold Delta tables copy to ADLS as Parquet.

In [0]:
# ============================================================
# GOLD DELTA TABLES → ADLS GOLD CONTAINER / PARQUET
# ============================================================

catalog = "vb_ecommerce"

# CORRECT GOLD CONTAINER
gold_adls_base = (
    "abfss://gold@adlsvbecommercedev.dfs.core.windows.net/"
    "ecommerce"
)

gold_tables = [
    "dim_customer",
    "dim_product",
    "dim_store",
    "fact_orders",
    "fact_sales",
    "fact_payments"
]

print("=" * 70)
print("GOLD → ADLS GOLD CONTAINER")
print("=" * 70)

for table in gold_tables:

    source_table = f"{catalog}.gold.{table}"
    target_path = f"{gold_adls_base}/{table}"

    print(f"\nExporting: {source_table}")

    df = spark.table(source_table)

    record_count = df.count()

    (
        df.write
        .format("parquet")
        .mode("overwrite")
        .save(target_path)
    )

    print(f"Records exported : {record_count}")
    print(f"ADLS path        : {target_path}")

print("\n" + "=" * 70)
print("GOLD → ADLS GOLD CONTAINER COMPLETED")
print("=" * 70)

GOLD → ADLS GOLD CONTAINER

Exporting: vb_ecommerce.gold.dim_customer
Records exported : 50
ADLS path        : abfss://gold@adlsvbecommercedev.dfs.core.windows.net/ecommerce/dim_customer

Exporting: vb_ecommerce.gold.dim_product
Records exported : 50
ADLS path        : abfss://gold@adlsvbecommercedev.dfs.core.windows.net/ecommerce/dim_product

Exporting: vb_ecommerce.gold.dim_store
Records exported : 50
ADLS path        : abfss://gold@adlsvbecommercedev.dfs.core.windows.net/ecommerce/dim_store

Exporting: vb_ecommerce.gold.fact_orders
Records exported : 50
ADLS path        : abfss://gold@adlsvbecommercedev.dfs.core.windows.net/ecommerce/fact_orders

Exporting: vb_ecommerce.gold.fact_sales
Records exported : 50
ADLS path        : abfss://gold@adlsvbecommercedev.dfs.core.windows.net/ecommerce/fact_sales

Exporting: vb_ecommerce.gold.fact_payments
Records exported : 50
ADLS path        : abfss://gold@adlsvbecommercedev.dfs.core.windows.net/ecommerce/fact_payments

GOLD → ADLS GOLD CONTAIN

##Export Silver to the data ADLS folder

In [0]:
# ============================================================
# SILVER DELTA TABLES → ADLS SILVER CONTAINER / PARQUET
# ============================================================

catalog = "vb_ecommerce"

silver_adls_base = (
    "abfss://silver@adlsvbecommercedev.dfs.core.windows.net/"
    "ecommerce"
)

silver_tables = [
    "customers",
    "products",
    "stores",
    "salesorders",
    "salesorderitems",
    "payments"
]

print("=" * 70)
print("SILVER → ADLS SILVER CONTAINER")
print("=" * 70)

for table in silver_tables:

    source_table = f"{catalog}.silver.{table}"
    target_path = f"{silver_adls_base}/{table}"

    df = spark.table(source_table)
    record_count = df.count()

    (
        df.write
        .format("parquet")
        .mode("overwrite")
        .save(target_path)
    )

    print(f"{table:<20} : {record_count} records")
    print(f"Path: {target_path}")

print("=" * 70)
print("SILVER EXPORT COMPLETED")
print("=" * 70)

SILVER → ADLS SILVER CONTAINER
customers            : 50 records
Path: abfss://silver@adlsvbecommercedev.dfs.core.windows.net/ecommerce/customers
products             : 50 records
Path: abfss://silver@adlsvbecommercedev.dfs.core.windows.net/ecommerce/products
stores               : 50 records
Path: abfss://silver@adlsvbecommercedev.dfs.core.windows.net/ecommerce/stores
salesorders          : 50 records
Path: abfss://silver@adlsvbecommercedev.dfs.core.windows.net/ecommerce/salesorders
salesorderitems      : 50 records
Path: abfss://silver@adlsvbecommercedev.dfs.core.windows.net/ecommerce/salesorderitems
payments             : 50 records
Path: abfss://silver@adlsvbecommercedev.dfs.core.windows.net/ecommerce/payments
SILVER EXPORT COMPLETED


##tesing codes

#=====================================================

In [0]:
%sql

SHOW STORAGE CREDENTIALS;

name,comment
cred_vb_ecommerce_adls,null
dbw_vb_ecommerce_dev,null


In [0]:
%sql

SHOW EXTERNAL LOCATIONS;

name,url,comment
dbw_vb_ecommerce_dev,abfss://unity-catalog-storage@dbstoragexoyocruayeybq.dfs.core.windows.net/7405617028985756,null
extloc_vb_ecommerce,abfss://landing@adlsvbecommercedev.dfs.core.windows.net/,null
extloc_vb_ecommerce_uc,abfss://bronze@adlsvbecommercedev.dfs.core.windows.net/unitycatalog,bronze/unitycatalog


In [0]:
bad_file = """
abfss://landing@adlsvbecommercedev.dfs.core.windows.net/
ecommerce/sql/customers/2026/08/29/customers_20260829_212303.parquet
""".replace("\n", "").replace(" ", "")

print(bad_file)

try:
    df_bad = spark.read.parquet(bad_file)

    print("File read successfully")
    print("Rows:", df_bad.count())
    display(df_bad)

except Exception as e:
    print("FILE READ FAILED")
    print(str(e)[:5000])

abfss://landing@adlsvbecommercedev.dfs.core.windows.net/ecommerce/sql/customers/2026/08/29/customers_20260829_212303.parquet
File read successfully
Rows: 1


CustomerID,FirstName,LastName,Email,Phone,City,State,Country,CreatedDate,ModifiedDate,IsDeleted
1,Arun,Kumar,arun@example.com,9876543210,Trivandrum,Kerala,India,2026-08-28T22:21:22.75Z,2026-08-29T21:19:12.986666Z,false


In [0]:
from pyspark.sql import functions as F

customer_path = """
abfss://landing@adlsvbecommercedev.dfs.core.windows.net/
ecommerce/sql/customers/2026/08/31
""".replace("\n", "").replace(" ", "")

df_today = spark.read.parquet(customer_path)

print("Today's Customer records:", df_today.count())

display(
    df_today.orderBy("CustomerID")
)

Today's Customer records: 52


CustomerID,FirstName,LastName,Email,Phone,City,State,Country,CreatedDate,ModifiedDate,IsDeleted
1,Arun,Kumar,arun@example.com,9876543210,Trivandrum,Kerala,India,2026-08-28T22:21:22.75Z,2026-08-29T21:19:12.986666Z,false
2,Priya,Nair,priya@example.com,9876543211,Kochi,Kerala,India,2026-08-28T22:21:22.75Z,2026-08-28T22:21:22.75Z,false
3,Rahul,Menon,rahul@example.com,9876543212,Chennai,Tamil Nadu,India,2026-08-28T22:21:22.75Z,2026-08-28T22:21:22.75Z,false
4,Sneha,Thomas,sneha@example.com,9876543213,Bengaluru,Karnataka,India,2026-08-28T22:21:22.75Z,2026-08-28T22:21:22.75Z,false
5,Vishnu,Raj,vishnu@example.com,9876543214,Coimbatore,Tamil Nadu,India,2026-08-28T22:21:22.75Z,2026-08-28T22:21:22.75Z,false
6,Customer6,User6,customer6@example.com,9876540006,Hyderabad,Kerala,India,2026-07-18T11:14:20.4Z,2026-07-18T11:14:20.4Z,false
7,Customer7,User7,customer7@example.com,9876540007,Mumbai,Tamil Nadu,India,2026-07-19T11:14:20.4Z,2026-07-19T11:14:20.4Z,false
8,Customer8,User8,customer8@example.com,9876540008,Trivandrum,Karnataka,India,2026-07-20T11:14:20.4Z,2026-07-20T11:14:20.4Z,false
9,Customer9,User9,customer9@example.com,9876540009,Kochi,Telangana,India,2026-07-21T11:14:20.4Z,2026-07-21T11:14:20.4Z,false
10,Customer10,User10,customer10@example.com,9876540010,Chennai,Maharashtra,India,2026-07-22T11:14:20.4Z,2026-07-22T11:14:20.4Z,false


In [0]:
customer_path = (
    "abfss://landing@adlsvbecommercedev.dfs.core.windows.net/"
    "ecommerce/sql/customers/2026/08/31"
)

files = dbutils.fs.ls(customer_path)

for f in files:
    print(f.name, f.size)

customers_20260831_130456.parquet 2221
customers_20260831_133627.parquet 4009


In [0]:
dbutils.fs.ls('/')

[FileInfo(path='dbfs:/Volume/', name='Volume/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/Volumes/', name='Volumes/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/databricks-datasets/', name='databricks-datasets/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/databricks-results/', name='databricks-results/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/volume/', name='volume/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/volumes/', name='volumes/', size=0, modificationTime=0)]

In [0]:
base = (
    "abfss://landing@adlsvbecommercedev.dfs.core.windows.net/"
    "ecommerce/sql/customers/2026/08/31/"
)

files = [
    "customers_20260831_130456.parquet",
    "customers_20260831_133627.parquet"
]

for file in files:
    path = base + file

    df = spark.read.parquet(path)

    print("=" * 60)
    print(f"FILE : {file}")
    print(f"ROWS : {df.count()}")
    print("=" * 60)

    display(
        df.select(
            "CustomerID",
            "FirstName",
            "LastName",
            "City",
            "ModifiedDate"
        ).orderBy("CustomerID")
    )

FILE : customers_20260831_130456.parquet
ROWS : 2


CustomerID,FirstName,LastName,City,ModifiedDate
49,Customer49,User49,Kochi,2026-08-30T11:14:20.4Z
50,Customer50,User50,Chennai,2026-08-31T11:14:20.4Z


FILE : customers_20260831_133627.parquet
ROWS : 50


CustomerID,FirstName,LastName,City,ModifiedDate
1,Arun,Kumar,Trivandrum,2026-08-29T21:19:12.986666Z
2,Priya,Nair,Kochi,2026-08-28T22:21:22.75Z
3,Rahul,Menon,Chennai,2026-08-28T22:21:22.75Z
4,Sneha,Thomas,Bengaluru,2026-08-28T22:21:22.75Z
5,Vishnu,Raj,Coimbatore,2026-08-28T22:21:22.75Z
6,Customer6,User6,Hyderabad,2026-07-18T11:14:20.4Z
7,Customer7,User7,Mumbai,2026-07-19T11:14:20.4Z
8,Customer8,User8,Trivandrum,2026-07-20T11:14:20.4Z
9,Customer9,User9,Kochi,2026-07-21T11:14:20.4Z
10,Customer10,User10,Chennai,2026-07-22T11:14:20.4Z


In [0]:
%sql DESCRIBE HISTORY vb_ecommerce.gold.dim_product;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
10,2026-08-31T10:00:15Z,1329982532534781,studywithmestudenthelp@gmail.com,MERGE,"Map(predicate -> [""(ProductID#7447 = ProductID#7431)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1888368466072533),0830-113514-b3wi3e6z,9,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 3966, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 5933, materializeSourceTimeMs -> 5, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2707, numTargetRowsUpdated -> 1, numOutputRows -> 1, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 1, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 3136)",null,Databricks-Runtime/17.3.x-scala2.13
9,2026-08-31T09:41:50Z,1329982532534781,studywithmestudenthelp@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true"",""delta.checkpoint.writeStatsAsStruct"":""true"",""delta.enableRowTracking"":""true"",""delta.checkpoint.writeStatsAsJson"":""false"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-2f0794cb-3db6-4297-8176-6417e6a5b889"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-42a1870a-537f-4b72-bdbc-4d8a1fd40493""}, statsOnLoad -> false)",null,List(1888368466072533),0830-113514-b3wi3e6z,8,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 4594, numDeletionVectorsRemoved -> 0, numOutputRows -> 50, numOutputBytes -> 4084)",null,Databricks-Runtime/17.3.x-scala2.13
8,2026-08-30T12:31:18Z,1329982532534781,studywithmestudenthelp@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> false, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3432410098042925),0830-113514-b3wi3e6z,7,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 12934, p25FileSize -> 4594, numDeletionVectorsRemoved -> 0, minFileSize -> 4594, numAddedFiles -> 1, maxFileSize -> 4594, p75FileSize -> 4594, p50FileSize -> 4594, numAddedBytes -> 4594)",null,Databricks-Runtime/17.3.x-scala2.13
7,2026-08-30T12:31:09Z,1329982532534781,studywithmestudenthelp@gmail.com,MERGE,"Map(predicate -> [""(cast(ProductID#8170 as bigint) = ProductID#8241L)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""NOT IsDeleted#8194"",""actionType"":""update""},{""predicate"":""IsDeleted#8194: boolean"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""predicate"":""NOT IsDeleted#8194"",""actionType"":""insert""}])",null,List(3432410098042925),0830-113514-b3wi3e6z,6,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 3, numTargetBytesAdded -> 12934, numTargetBytesRemoved -> 4464, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 5, executionTimeMs -> 3493, materializeSourceTimeMs -> 384, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1203, numTargetRowsUpdated -> 5, numOutputRows -> 6, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 6, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1880)",null,Databricks-Runtime/17.3.x-scala2.13
6,20

In [0]:
%sql
SELECT *
FROM vb_ecommerce.gold.dim_product VERSION AS OF 10;

ProductID,ProductName,Category,Price,StockQuantity,ProductStatus,CreatedDate,ModifiedDate
2,Mobile Phone,Electronics,30000.00,100,Active,2026-08-28T22:23:46.173333Z,2026-08-28T22:23:46.173333Z
3,Headphones,Electronics,2500.00,200,Active,2026-08-28T22:23:46.173333Z,2026-08-28T22:23:46.173333Z
4,Running Shoes,Sports,4500.00,75,Active,2026-08-28T22:23:46.173333Z,2026-08-28T22:23:46.173333Z
5,Backpack,Accessories,1800.00,120,Active,2026-08-28T22:23:46.173333Z,2026-08-28T22:23:46.173333Z
6,Keyboard 6,Accessories,1800.00,62,Active,2026-07-18T11:14:20.54Z,2026-07-18T11:14:20.54Z
7,Monitor 7,Sports,13050.00,69,Active,2026-07-19T11:14:20.54Z,2026-07-19T11:14:20.54Z
8,Tablet 8,Computers,69000.00,76,Active,2026-07-20T11:14:20.54Z,2026-07-20T11:14:20.54Z
9,Wireless Mouse 9,Mobile,32250.00,83,Active,2026-07-21T11:14:20.54Z,2026-07-21T11:14:20.54Z
10,Laptop 10,Electronics,3000.00,90,Active,2026-07-22T11:14:20.54Z,2026-07-22T11:14:20.54Z
11,Mobile Phone 11,Accessories,5600.00,97,Active,2026-07-23T11:14:20.54Z,2026-07-23T11:14:20.54Z
